# Store Sales
## Feature Engineering
* **Goal:** Prepare and transform the dataset for property price prediction

This notebook prepares the cleaned dataset for machine learning models. 

In [1]:
# Import libraries

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [2]:
# Define path

data_path = Path('../data')

# Find CSV
files = list(data_path.glob('*.csv'))

# Load files into dictonary
dataframes = {}

for file in files:
    name = file.stem
    dataframes[name] = pd.read_csv(file)

In [3]:
# Convert date cols to datatime format

for name, df in dataframes.items():
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])

In [4]:
# Display the shape of each dataset

for name, df in dataframes.items():
    print(f'{name}: {df.shape}')

test: (28512, 5)
train: (3000888, 6)
transactions: (83488, 3)
oil: (1218, 2)
holidays_events: (350, 6)
sample_submission: (28512, 2)
stores: (54, 5)


In [5]:
# Create a copy of the train data 
train = dataframes['train'].copy()

train.head()

,id,date,store_nbr,family,sales,onpromotion
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0
1,1,2013-01-01,1,BABY CARE,0.0,0
2,2,2013-01-01,1,BEAUTY,0.0,0
3,3,2013-01-01,1,BEVERAGES,0.0,0
4,4,2013-01-01,1,BOOKS,0.0,0


In [6]:
# Display shape, cols and missing values
print('Shape:', train.shape)
print(train.columns.tolist())
print(train.isna().sum())


Shape: (3000888, 6)
['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion']
id             0
date           0
store_nbr      0
family         0
sales          0
onpromotion    0
dtype: int64


In [7]:
# Add Store information to train dataset
train = train.merge(
    dataframes['stores'], 
    on='store_nbr', 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}')
print(train.head())

Shape: (3000888, 10)
   id       date  store_nbr      family  sales  onpromotion   city      state  \
0   0 2013-01-01          1  AUTOMOTIVE    0.0            0  Quito  Pichincha   
1   1 2013-01-01          1   BABY CARE    0.0            0  Quito  Pichincha   
2   2 2013-01-01          1      BEAUTY    0.0            0  Quito  Pichincha   
3   3 2013-01-01          1   BEVERAGES    0.0            0  Quito  Pichincha   
4   4 2013-01-01          1       BOOKS    0.0            0  Quito  Pichincha   

  type  cluster  
0    D       13  
1    D       13  
2    D       13  
3    D       13  
4    D       13  


In [8]:
# Add transaction data
train = train.merge(
    dataframes['transactions'], 
    on=['date', 'store_nbr'], 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}')

# Check missing values for transaction
print(f'Missing transactions: {train['transactions'].isna().sum()}')

Shape: (3000888, 11)
Missing transactions: 245784


In [9]:
# Add oil price datas
train = train.merge(
    dataframes['oil'], 
    on='date', 
    how='left'
)

# Check the result
print(f'Shape: {train.shape}]')
print(f'Missing oil prices: {train['dcoilwtico'].isna().sum()}')

Shape: (3000888, 12)]
Missing oil prices: 928422


In [10]:
# Extract time-based features from date cols
train['year'] = train['date'].dt.year
train['month'] = train['date'].dt.month
train['day'] = train['date'].dt.day
train['day_of_week'] = train['date'].dt.dayofweek

# Display new features
train[['date', 'year', 'month', 'day', 'day_of_week']].head()

,date,year,month,day,day_of_week
0,2013-01-01,2013,1,1,1
1,2013-01-01,2013,1,1,1
2,2013-01-01,2013,1,1,1
3,2013-01-01,2013,1,1,1
4,2013-01-01,2013,1,1,1


In [11]:
# Identify weekends
train['is_weekend'] = train['day_of_week'].isin([5, 6]).astype(int)

# Extract quarter
train['quarter'] = train['date'].dt.quarter

# Display new featrues
train[['date', 'day_of_week', 'is_weekend', 'quarter']].head()

,date,day_of_week,is_weekend,quarter
0,2013-01-01,1,0,1
1,2013-01-01,1,0,1
2,2013-01-01,1,0,1
3,2013-01-01,1,0,1
4,2013-01-01,1,0,1


In [12]:
# Get unique dates from holiday and events dataset
special_dates = dataframes['holidays_events']['date'].unique()

# Identiy special dates
train['is_special_date'] = train['date'].isin(special_dates).astype(int)

# Check new features
print(train['is_special_date'].value_counts())

is_special_date
0    2551824
1     449064
Name: count, dtype: int64


In [13]:
# Check missing values
missing_values = train.isna().sum()

missing_percentage = (missing_values/len(train)) * 100

missing_summary = pd.DataFrame({
    'missing_values': missing_values,
    'missing_percentage': missing_percentage
})

missing_summary = missing_summary[missing_summary['missing_values'] > 0]
missing_summary

,missing_values,missing_percentage
transactions,245784,8.190376
dcoilwtico,928422,30.938242


In [14]:
# Sort by date
train = train.sort_values('date')

train['dcoilwtico'] = train['dcoilwtico'].ffill()

# Check remaining missing values
print(f'Missing oil prices: {train['dcoilwtico'].isna().sum()}')

Missing oil prices: 1782


In [15]:
# Show missing oil prices
missing_oil = train[train['dcoilwtico'].isna()]

# Check date range
print(f'First missing date: {missing_oil['date'].min()}')
print(f'Last missing date: {missing_oil['date'].max()}')
print(f'Number of missing rows: {len(missing_oil)}')

First missing date: 2013-01-01 00:00:00
Last missing date: 2013-01-01 00:00:00
Number of missing rows: 1782


In [16]:
# Fill remaining missing oil prices
train['dcoilwtico'] = train['dcoilwtico'].bfill()

# Check result
print(f'Missing oil prices: {train['dcoilwtico'].isna().sum()}')

Missing oil prices: 0


### Handling Missing Oil Prices

The oil price data contained missing values, including missing values at the beginning of the time series.

First, forward fill (`ffill()`) was used to fill missing values with the last available oil price. The remaining missing values occurred on January 1, 2013, when no previous oil price was available.

Backward fill (`bfill()`) was then applied to fill the remaining missing values using the next available oil price.

After this process, no missing values remained in the `dcoilwtico` column.

**Note:** Backward fill uses a future observation to fill a historical value. This decision should be considered when evaluating the model to avoid potential data leakage.

In [17]:
# Check oil prices
print(train['dcoilwtico'].isna().sum())
print(train['dcoilwtico'].describe())

0
count    3.000888e+06
mean     6.792490e+01
std      2.566913e+01
min      2.619000e+01
25%      4.637750e+01
50%      5.341000e+01
75%      9.572000e+01
max      1.106200e+02
Name: dcoilwtico, dtype: float64
